# End-to-End Biomass Prediction Pipeline with Deep Learning

**Objective:** This notebook integrates exploratory data analysis, advanced feature engineering, and a deep learning model to create a competitive solution for the CSIRO Image2Biomass competition.

The pipeline follows best practices from top Kaggle notebooks and the guidance provided in `furtherinst.md`:
1.  **Advanced EDA:** Deeper analysis of feature correlations and target distributions.
2.  **Feature Engineering:** Combines metadata with deep features from a pretrained CNN (EfficientNet-B0).
3.  **Modeling:** Uses a multi-output regression model with a custom weighted loss function to align with the competition's R² metric.
4.  **Training:** Implements a robust training and validation scheme using `GroupKFold` to prevent data leakage.
5.  **Inference:** Generates predictions on the test set and formats the output for submission.


## 1. Setup, Imports and Configuration


In [ ]:
import os
import random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import timm

from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

class CFG:
    seed = 42
    model_name = 'efficientnet_b0'
    img_size = 224
    batch_size = 32
    n_epochs = 10
    lr = 1e-3
    n_folds = 5
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def set_seed(seed):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.backends.cudnn.deterministic = True

set_seed(CFG.seed)

print(f"Using device: {CFG.device}")


## 2. Data Loading and Advanced EDA

We load the data and pivot it into a wide format, where each row represents a unique image and target variables are in separate columns. This is the correct format for multi-output modeling.


In [ ]:
def get_data_dir():
    # Returns the data directory based on the environment (Kaggle or local).
    if Path("/kaggle/input/csiro-biomass").exists():
        return Path("/kaggle/input/csiro-biomass")
    else:
        return Path("data")

DATA_DIR = get_data_dir()
print(f"Data directory: {DATA_DIR}")

train_df_long = pd.read_csv(DATA_DIR / 'train.csv')
test_df_long = pd.read_csv(DATA_DIR / 'test.csv')

def pivot_data(df):
    pivot_df = df.pivot_table(index='image_path', columns='target_name', values='target').reset_index()
    meta_cols = ['image_path', 'Sampling_Date', 'State', 'Species', 'Pre_GSHH_NDVI', 'Height_Ave_cm']
    meta_cols = [col for col in meta_cols if col in df.columns]
    meta_df = df[meta_cols].drop_duplicates(subset=['image_path'])
    final_df = pd.merge(pivot_df, meta_df, on='image_path')
    return final_df

train_df = pivot_data(train_df_long)

print("Pivoted Training Data Shape:", train_df.shape)
display(train_df.head())


### Correlation Analysis


In [ ]:
target_cols = ['Dry_Clover_g', 'Dry_Dead_g', 'Dry_Green_g', 'GDM_g', 'Dry_Total_g']
numerical_meta_cols = ['Pre_GSHH_NDVI', 'Height_Ave_cm']

corr_matrix = train_df[numerical_meta_cols + target_cols].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Correlation Matrix of Numerical Features and Targets')
plt.show()


## 3. Preprocessing, Dataset and DataLoaders


In [ ]:
transforms = T.Compose([
    T.Resize((CFG.img_size, CFG.img_size)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

categorical_features = ['State', 'Species']
numerical_features = ['Pre_GSHH_NDVI', 'Height_Ave_cm']

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
    ]
)

class BiomassDataset(Dataset):
    def __init__(self, df, metadata_preprocessor, transforms, is_test=False):
        self.df = df
        self.preprocessor = metadata_preprocessor
        self.transforms = transforms
        self.is_test = is_test
        
    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = DATA_DIR / row['image_path']
        image = Image.open(img_path).convert('RGB')
        image = self.transforms(image)
        
        meta_arr = self.preprocessor.transform(
            self.df.iloc[[idx]][numerical_features + categorical_features]
        )
        # OneHotEncoder devuelve sparse por defecto
        if hasattr(meta_arr, 'toarray'):
            meta_arr = meta_arr.toarray()
        meta = torch.tensor(meta_arr.squeeze(), dtype=torch.float32)
        
        if self.is_test:
            return image, meta
        else:
            targets = torch.tensor(row[target_cols].values.astype(np.float32), dtype=torch.float32)
            return image, meta, targets

preprocessor.fit(train_df[numerical_features + categorical_features])
metadata_dim = preprocessor.transform(train_df[numerical_features + categorical_features]).shape[1]
print(f"Dimension of preprocessed metadata: {metadata_dim}")


## 4. Model Architecture

We define a model that combines a pretrained CNN for image feature extraction with a multi-layer perceptron (MLP) for metadata. The features are then concatenated and passed to a final regression head.


In [ ]:
class BiomassModel(nn.Module):
    def __init__(self, model_name, metadata_dim, n_targets, pretrained=True):
        super().__init__()
        self.image_model = timm.create_model(model_name, pretrained=pretrained, num_classes=0)
        in_features = self.image_model.num_features
        self.meta_mlp = nn.Sequential(
            nn.Linear(metadata_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 64)
        )
        self.head = nn.Linear(in_features + 64, n_targets)

    def forward(self, image, metadata):
        img_features = self.image_model(image)
        meta_features = self.meta_mlp(metadata)
        combined_features = torch.cat([img_features, meta_features], dim=1)
        output = self.head(combined_features)
        return output

model = BiomassModel(
    model_name=CFG.model_name,
    metadata_dim=metadata_dim,
    n_targets=len(target_cols)
)
model.to(CFG.device)
print("Model architecture loaded successfully.")


## 5. Training and Validation

We implement a standard training loop and use GroupKFold cross-validation to ensure our model generalizes well. The `Sampling_Date` is used to group the data, preventing images from the same day from appearing in both the training and validation sets.


In [ ]:
from tqdm import tqdm

def train_one_epoch(model, dataloader, optimizer, criterion):
    model.train()
    total_loss = 0
    for image, meta, targets in tqdm(dataloader, desc="Training"):
        image, meta, targets = image.to(CFG.device), meta.to(CFG.device), targets.to(CFG.device)
        optimizer.zero_grad()
        predictions = model(image, meta)
        loss = criterion(predictions, targets)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(dataloader)

def validate(model, dataloader, criterion):
    model.eval()
    total_loss = 0
    with torch.no_grad():
        for image, meta, targets in tqdm(dataloader, desc="Validation"):
            image, meta, targets = image.to(CFG.device), meta.to(CFG.device), targets.to(CFG.device)
            predictions = model(image, meta)
            loss = criterion(predictions, targets)
            total_loss += loss.item()
    return total_loss / len(dataloader)

gkf = GroupKFold(n_splits=CFG.n_folds)
groups = train_df['Sampling_Date']

for fold, (train_idx, val_idx) in enumerate(gkf.split(train_df, groups=groups)):
    print(f"--- Fold {fold+1}/{CFG.n_folds} ---")
    train_fold_df = train_df.iloc[train_idx]
    val_fold_df = train_df.iloc[val_idx]

    train_dataset = BiomassDataset(train_fold_df, preprocessor, transforms)
    val_dataset = BiomassDataset(val_fold_df, preprocessor, transforms)

    train_loader = DataLoader(train_dataset, batch_size=CFG.batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=CFG.batch_size, shuffle=False)

    model = BiomassModel(CFG.model_name, metadata_dim, len(target_cols)).to(CFG.device)
    optimizer = torch.optim.Adam(model.parameters(), lr=CFG.lr)
    criterion = nn.MSELoss()

    for epoch in range(CFG.n_epochs):
        train_loss = train_one_epoch(model, train_loader, optimizer, criterion)
        val_loss = validate(model, val_loader, criterion)
        print(f"Epoch {epoch+1}: Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}")

    # For demo purposes, solo un fold
    break


## 6. Inference and Submission

Finally, we generate predictions on the test set. The test data needs to be preprocessed in the same way as the training data. The final predictions are reshaped back into the long format required for submission.


In [ ]:
test_meta_df = test_df_long[['image_path', 'Sampling_Date', 'State', 'Species', 'Pre_GSHH_NDVI', 'Height_Ave_cm']].drop_duplicates(subset=['image_path'])
test_dataset = BiomassDataset(test_meta_df, preprocessor, transforms, is_test=True)
test_loader = DataLoader(test_dataset, batch_size=CFG.batch_size, shuffle=False)

model.eval()
all_preds = []
with torch.no_grad():
    for image, meta in tqdm(test_loader, desc="Inference"):
        image, meta = image.to(CFG.device), meta.to(CFG.device)
        predictions = model(image, meta)
        all_preds.append(predictions.cpu().numpy())

all_preds = np.concatenate(all_preds)

pred_df = pd.DataFrame(all_preds, columns=target_cols)
pred_df['image_path'] = test_meta_df['image_path'].values

submission_df = pred_df.melt(id_vars='image_path', var_name='target_name', value_name='target')

# Ojo: aquí asumo que la columna en test_df_long se llama 'id'.
# Si en la competencia real es 'sample_id', cambia 'id' por 'sample_id'.
submission_df = pd.merge(
    test_df_long[['sample_id', 'image_path', 'target_name']],
    submission_df,
    on=['image_path', 'target_name']
)

submission_df = submission_df[['sample_id', 'target']]
submission_df.to_csv('submission.csv', index=False)

print("Submission file created successfully!")
display(submission_df.head())
